## Example: Adding new datasets to ANYI and the ANYI Browser

* If you have a new dataset that you want to consider in the context of the yeast interactome or test its correlation with parameters already in ANYI, you can quickly generate an updated .pkl file based on the current release
* The steps below explain, using simulated example data, how to add new annotations to ANYI for your own research purposes
* We also explain how to create an update `json` configuration file for the ANYI Browser to enable coloring the network by a new parameter

**Important note**: to guarantee compatibility as software versions drift over time, make sure you are using the base environment within the ANYI container from DockerHub. The updated version of ANYI that you create will also need to be utilized from the ANYI environment to guarantee stability. 

### Step 0 - Load libraries
* Execute the code cell below to load required libraries

In [1]:
import pandas as pd
import numpy as np
import string

#### Step 1 - Load the current database
* Load the current database build

In [2]:
nodes = pd.read_pickle("../data/nodes.pkl")

#### Step 2 - Prepare your new data
* The key to adding new data to ANYI is to relate one of two unique identifiers to your data:
    * An SGD unique identifier of the form `YCL008C` (`node` within ANYI)
    * A UniProt Knowledge Base accession number like `P25604` (`UniProtKB-AV` within ANYI)
* In this example, we generate example data with Python and provide an explanation of how to prepare your own data to add to ANYI

In [3]:
rng = np.random.default_rng(seed=0)

n = len(nodes)

# take the nodes column from nodes and generate five columns of random data
new_data = pd.DataFrame({
    "node": nodes["node"].values,
    "new_column1": rng.normal(loc=0, scale=1, size=n),
    "new_column2": rng.normal(loc=0, scale=1, size=n),
    "new_column3": rng.normal(loc=0, scale=1, size=n),
    "new_column4": rng.normal(loc=0, scale=1, size=n),
    "new_column5": rng.normal(loc=0, scale=1, size=n),
})

# simulate real data that will not have exactly 
# 1 to 1 matching identifiers with ANYI

# drop 150 random rows
rows_to_drop = rng.choice(new_data.index, size=150, replace=False)
new_data = new_data.drop(index=rows_to_drop)

# add 200 rows with identifiers not in ANYI
alphabet = np.array(list(string.ascii_uppercase + string.digits))
suffixes = rng.choice(alphabet, size=(200, 6))
node_ids = np.array(["X" + "".join(row) for row in suffixes])
new_data2 = pd.DataFrame({
    "node": node_ids,
    "new_column1": rng.normal(loc=0, scale=1, size=200),
    "new_column2": rng.normal(loc=0, scale=1, size=200),
    "new_column3": rng.normal(loc=0, scale=1, size=200),
    "new_column4": rng.normal(loc=0, scale=1, size=200),
    "new_column5": rng.normal(loc=0, scale=1, size=200),
})

# create a single pd.DataFrame to serve as our dummy data
data = pd.concat([new_data, new_data2], ignore_index=True)

# print a summary
data.info()

# if loading data from a file, you can load it like this for a .csv file:
#new_data = pd.read_csv("path/to/your/file.csv")

# or like this for a .tsv file:
# new_data = pd.read_csv("path/to/your/file.tsv", sep="\t")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3977 entries, 0 to 3976
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   node         3977 non-null   object 
 1   new_column1  3977 non-null   float64
 2   new_column2  3977 non-null   float64
 3   new_column3  3977 non-null   float64
 4   new_column4  3977 non-null   float64
 5   new_column5  3977 non-null   float64
dtypes: float64(5), object(1)
memory usage: 186.6+ KB


* ANYI has data for 3,927 nodes. We removed 150 at random and then added 200, so we expect 3,977 rows (one per node) in `data`, the `pd.DataFrame` object that contains our dummy data we are adding to ANYI.
* in practice, you can prepare your data in a spreadsheet (e.g., Google Sheets, Microsoft Excel) and then save a comma- or tab-separated file
    * as mentioned above, it will be critical to include one of the two unique identifiers
* you can then read this file into Python using Pandas and continue with the steps below (see comments in the Step 2 code cell)

#### Step 3 - Joining the new data into ANYI
* we now have `nodes` containing the current set of annotations in ANYI and `data` containing the dummy data we are adding to ANYI
* the cell 

In [4]:
# ANYI has 155 columns
nodes.info()

# left merge preserves the "shape" of ANYI; we only have interaction
# data for these 3,927 proteins
merged = nodes.merge(data, on="node", how="left")

# the new version has 160 columns
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3927 entries, 0 to 3926
Columns: 155 entries, node to meltome-fit-R2
dtypes: bool(2), float64(56), int64(39), object(58)
memory usage: 4.6+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3927 entries, 0 to 3926
Columns: 160 entries, node to new_column5
dtypes: bool(2), float64(61), int64(39), object(58)
memory usage: 4.7+ MB


* the output from the above `.info()` calls indicates that our merged dataset `merged` contains 160 columns and 3,927 rows
    * the number of rows remains 3,927
    * the number of columns increases by 5 to 160 (the `node` columns appears in both `nodes` and `data` and is not duplicated in `merged`, so we cleanly add only the 5 new data columns `new_column1`, ..., `new_column2`
* run the code cell below and inspect the output to confirm the behavior of the merge command

In [12]:
temp = merged[["node", "new_column1", "new_column2", "new_column3", "new_column4", "new_column5"]]

temp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3927 entries, 0 to 3926
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   node         3927 non-null   object 
 1   new_column1  3777 non-null   float64
 2   new_column2  3777 non-null   float64
 3   new_column3  3777 non-null   float64
 4   new_column4  3777 non-null   float64
 5   new_column5  3777 non-null   float64
dtypes: float64(5), object(1)
memory usage: 184.2+ KB


* the output of this cell indicates that we have added 3,777 new pieces of information in each of 5 columns, which makes sense if we review our processing steps and the behvaior of a left merge:
    * we started with five columns of information for the same 3,927 proteins/nodes in ANYI
    * we then removed 150 proteins/nodes, leaving 3,777 rows in the new dataset
    * finally, we added 200 rows with identifiers of the form `X123456`, giving us 3,977 rows in `data`
* the identifiers for the 200 rows added at the final step above do not map to anything in the ANYI nodes columnm, so they are simply dropped
* the five new columns for 3,777 rows of `data` with identifiers in ANYI are added to the `pd.DataFrame` merged and can now be saved for later use

In [16]:
output_name = "updated-nodes.pkl"
merged.to_pickle(output_name)

#### Step 4 - updating the ANYI Browser to use your new database version
* we have now integrated five new columns into ANYI. In order to interact with these new columns from the ANYI browser, we need to do two key things:
    1) update your configuration file to target the `updated-nodes.pkl` instead of `nodes.pkl` and display information on the new columns
    2) update `ANYI-browser.ipynb` to target your new configuration file.

##### Step 4a - update your configuration file

* the file at the path (from the repository root) `docker/ANYI-browser/config/base_config.json` is the base configuration file for the default ANYI display
* the file at the path `docker/ANYI-browser/config/test_config.json` is an updated configuration file that targets `docker/ANYI-browser/examples/updated-nodes.pkl` instead of `docker/ANYI-browser/data/nodes.pkl`

Here is our `base_config.json` file:

```json
{
  "data": {
    "nodes_file": "nodes.pkl",
    "edges_file": "edges.csv"
  },
  "field_labels": {
    "node": "Yeast ORF",
    "UniProtKB-AC": "UniProt Accession",
    "ProteinName": "Protein name",
    "degree_centrality_percentile": "Degree centrality (percentile)",
    "betweenness_centrality_percentile": "Betweenness centrality (percentile)",
    "DeepTMHMM_class": "DeepTMHMM (topology class)",
    "parsed_functions": "UniProt function",
    "median_molecules_per_cell":"Median molecules per cell",
    "Villen_halflife_min":"Half-life (min)",
    "meltome-melting-point":"Melting point (Meltome Atlas, °C)"
  },
  "network_coloring": {
    "metrics": [
      "degree_centrality",
      "betweenness_centrality",
      "median_molecules_per_cell",
      "Villen_halflife_min",
      "meltome-melting-point"
    ]
  },
  "annotation_sections": [
    {
      "title": "Identifiers",
      "fields": ["node", "UniProtKB-AC", "ProteinName"]
    },
    {
      "title": "Function",
      "fields": ["parsed_functions"]
    },
    {
      "title": "Proteostasis",
      "fields": ["median_molecules_per_cell", "Villen_halflife_min", "meltome-melting-point"]
    },
    {
      "title": "Network",
      "fields": ["degree_centrality_percentile", "betweenness_centrality_percentile"]
    },
    {
      "title": "Topology",
      "fields": ["DeepTMHMM_class"]
    }
  ]
}
```

It contains four sections:
* `data` - defines the node and edge data used by the browser
* `field_labels` - defines mappings between the raw column names in ANYI and descriptive display names
* `network_coloring` - a list of ANYI node names that will be converted to percentile scales and then made available as coloring methods for ego networks in ANYI
* `annotation_sections` - a list of sections (e.g., Identifiers, Network) and the annotations that will be displayed in them. This section also controls the annotations that appear in the hover text

Let's define our goal. We want to:
* use a new `nodes.pkl` file version with five new columns, 
* add each of these columns to the displayed annotations, 
* convert one of the annotations (`new_column3` for demonstration purposes) to a percentile scale and make it available as a coloring method for the ego network. 

Our new configuration file `docker/ANYI-browser/config/test_config.json` is:

```json
{
  "data": {
    "nodes_file": "examples/updated-nodes.pkl",
    "edges_file": "edges.csv"
  },
  "field_labels": {
    "node": "Yeast ORF",
    "UniProtKB-AC": "UniProt Accession",
    "ProteinName": "Protein name",
    "degree_centrality_percentile": "Degree centrality (percentile)",
    "betweenness_centrality_percentile": "Betweenness centrality (percentile)",
    "DeepTMHMM_class": "DeepTMHMM (topology class)",
    "parsed_functions": "UniProt function",
    "median_molecules_per_cell":"Median molecules per cell",
    "Villen_halflife_min":"Half-life (min)",
    "meltome-melting-point":"Melting point (Meltome Atlas, °C)",
    "new_column1":"First new data type",
    "new_column2":"Second new data type",
    "new_column3":"Third new data type",
    "new_column3_percentile":"Third new data type (percentile)",
    "new_column4":"Fourth new data type",
    "new_column5":"Fifth new data type"
  },
  "network_coloring": {
    "metrics": [
      "degree_centrality",
      "betweenness_centrality",
      "median_molecules_per_cell",
      "Villen_halflife_min",
      "meltome-melting-point",
      "new_column3"
    ]
  },
  "annotation_sections": [
    {
      "title": "Identifiers",
      "fields": ["node", "UniProtKB-AC", "ProteinName"]
    },
    {
      "title": "Function",
      "fields": ["parsed_functions"]
    },
    {
      "title": "Proteostasis",
      "fields": ["median_molecules_per_cell", "Villen_halflife_min", "meltome-melting-point"]
    },
    {
      "title": "Network",
      "fields": ["degree_centrality_percentile", "betweenness_centrality_percentile"]
    },
    {
      "title": "Topology",
      "fields": ["DeepTMHMM_class"]
    },
    {
      "title": "New Annotations",
      "fields": ["new_column1", "new_column2", "new_column3", "new_column3_percentile", "new_column4", "new_column5"]
    }
  ]
}
```

We have made four distinct changes:
* First, we changed `nodes_file` to a relative path from `ANYI-browser.ipynb` to `updated-nodes.pkl` rather than to the default .pkl
* Second, we added `new_column1`, `new_column2`, `new_column3`, `new_column3_percentile`, `new_column`, `new_column`, to `field_labels` with names to be output in browser
* Third, we added `new_column3` to the `network_coloring` section
* Fourth, we added an `annotation_sections` with title `New Annotations` to display each of the new pieces of information
